# Taller modelos de regresión lineal regularizados

Con el archivo de datos suministrado, va a crear un modelo **lineal** que prediga la variable **Weight**.

Información sobre este archivo, incluido el diccionario de datos, la puede obtener en [Kaggle](https://www.kaggle.com/datasets/samruddhim/olympics-althlete-events-analysis).

In [50]:
from pathlib import Path
import pandas as pd

#aqui le decimos a python en que carpeta estan los datos
DATA_DIR = Path().resolve().parent / "data" / "raw"
data_file = "athlete_events.csv"
data_path = DATA_DIR / data_file
df = pd.read_csv(data_path) 
df.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


Haga un análisis exploratorio y descarte variables irrelevantes.

Elimine los registros duplicados.

Elimine datos nulos de la variable objetivo.

Impute a los datos nulos de la variable **Medal** la categoría **No medal**.

In [51]:
#Eliminar Games es variable redundante
df.drop(columns=["Games"], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 14 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  object 
 2   Sex     271116 non-null  object 
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  object 
 7   NOC     271116 non-null  object 
 8   Year    271116 non-null  int64  
 9   Season  271116 non-null  object 
 10  City    271116 non-null  object 
 11  Sport   271116 non-null  object 
 12  Event   271116 non-null  object 
 13  Medal   39783 non-null   object 
dtypes: float64(3), int64(2), object(9)
memory usage: 29.0+ MB


In [52]:
df.describe()

,ID,Age,Height,Weight,Year
count,271116.000000,261642.000000,210945.000000,208241.000000,271116.000000
mean,68248.954396,25.556898,175.338970,70.702393,1978.378480
std,39022.286345,6.393561,10.518462,14.348020,29.877632
min,1.000000,10.000000,127.000000,25.000000,1896.000000
25%,34643.000000,21.000000,168.000000,60.000000,1960.000000
50%,68205.000000,24.000000,175.000000,70.000000,1988.000000
75%,102097.250000,28.000000,183.000000,79.000000,2002.000000
max,135571.000000,97.000000,226.000000,214.000000,2016.000000


In [53]:
#ID debe ser variable categorica
df['ID'] = df['ID'].astype('object')

**Variables deben descartarse**

- ID y Names son casi claves primaras no predicen la variable weight

- Team, Event, NOC, Sport y City tienen alta cardinalidad.Para procesarlar se usa targetEncoder

In [54]:
df = df.drop(columns=['ID', 'Name'])

In [55]:
df.describe(include="object").T

,count,unique,top,freq
Sex,271116,2,M,196594
Team,271116,1184,United States,17847
NOC,271116,230,USA,18853
Season,271116,2,Summer,222552
City,271116,42,London,22426
Sport,271116,66,Athletics,38624
Event,271116,765,Football Men's Football,5733
Medal,39783,3,Gold,13372


In [56]:
#proporcion valores nulos %
df.isnull().sum()*100/len(df)

Sex        0.000000
Age        3.494445
Height    22.193821
Weight    23.191180
Team       0.000000
NOC        0.000000
Year       0.000000
Season     0.000000
City       0.000000
Sport      0.000000
Event      0.000000
Medal     85.326207
dtype: float64

In [57]:
#La variable objetivo tiene datos nulos; estos deben descartarse.
df.dropna(subset=['Weight'], inplace=True)


 En el caso de la variable **Medal**, se imputa los datos nulos el valor **No medal**.

In [58]:
df = df.fillna({'Medal': 'No medal'})
df.isnull().sum()*100 / len(df)
print(len(df))

208241


In [60]:
df['Medal'].value_counts()

Medal
No medal    175984
Gold         10167
Bronze       10148
Silver        9866
Name: count, dtype: int64

Al descartar los nulos de Weight, la proporcion de datos nulos de Height y Age son bajos entonces pueden descartarse

In [59]:
df.dropna(subset=['Age','Height'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 206165 entries, 0 to 271115
Data columns (total 12 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Sex     206165 non-null  object 
 1   Age     206165 non-null  float64
 2   Height  206165 non-null  float64
 3   Weight  206165 non-null  float64
 4   Team    206165 non-null  object 
 5   NOC     206165 non-null  object 
 6   Year    206165 non-null  int64  
 7   Season  206165 non-null  object 
 8   City    206165 non-null  object 
 9   Sport   206165 non-null  object 
 10  Event   206165 non-null  object 
 11  Medal   206165 non-null  object 
dtypes: float64(3), int64(1), object(8)
memory usage: 20.4+ MB


In [61]:
#Se eliminan duplicados
df = df.drop_duplicates()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 205846 entries, 0 to 271115
Data columns (total 12 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Sex     205846 non-null  object 
 1   Age     205846 non-null  float64
 2   Height  205846 non-null  float64
 3   Weight  205846 non-null  float64
 4   Team    205846 non-null  object 
 5   NOC     205846 non-null  object 
 6   Year    205846 non-null  int64  
 7   Season  205846 non-null  object 
 8   City    205846 non-null  object 
 9   Sport   205846 non-null  object 
 10  Event   205846 non-null  object 
 11  Medal   205846 non-null  object 
dtypes: float64(3), int64(1), object(8)
memory usage: 20.4+ MB


## EDA

In [62]:
#Categorias de variables categoricas no descartadas
cat_vars = df.select_dtypes(include='object').columns
for var in cat_vars:
    print(f'{var}:\n {df[var].unique()} unique values \n')

Sex:
 ['M' 'F'] unique values 

Team:
 ['China' 'Netherlands' 'United States' 'Finland' 'Norway' 'Romania'
 'Estonia' 'France' 'Spain' 'Egypt' 'Bulgaria' 'Italy' 'Azerbaijan'
 'Russia' 'Argentina' 'Cuba' 'Belarus' 'Greece' 'Cameroon' 'Mexico'
 'Soviet Union' 'Nicaragua' 'Hungary' 'Nigeria' 'Algeria' 'Kuwait'
 'Bahrain' 'Pakistan' 'Iraq' 'Lebanon' 'Qatar' 'Malaysia' 'Iran' 'Canada'
 'Ireland' 'Australia' 'South Africa' 'Morocco' 'Eritrea' 'Tanzania'
 'Jordan' 'Sudan' 'Tunisia' 'Libya' 'Belgium' 'Djibouti' 'Comoros'
 'Kazakhstan' 'Brunei' 'Saudi Arabia' 'Maldives' 'Ethiopia' 'Indonesia'
 'Philippines' 'Uzbekistan' 'United Arab Emirates' 'Kyrgyzstan'
 'Tajikistan' 'Unified Team' 'Japan' 'Japan-1' 'Brazil' 'West Germany'
 'East Germany' 'Germany' 'Israel' 'Sweden' 'United States Virgin Islands'
 'Turkey' 'Sri Lanka' 'Armenia' "Cote d'Ivoire" 'Kenya' 'Benin' 'France-1'
 'Ukraine' 'Ghana' 'Somalia' 'Latvia' 'Syria' 'Great Britain' 'Chile'
 'Switzerland' 'India' 'Poland' 'Costa Rica' 'Panama'

Cree la matriz de características y el vector objetivo. 

Parta los datos en conjuntos de entrenamiento y prueba en una proporción 80/20.

Especifique, usando `ColumnTransformer` y `Pipeline`, el procesamiento que le va a realizar a los características de su modelo. Utilice `TargetEncoder` para codificar las variables de alta cardinalidad, mayor a 5.

In [67]:
#Eliminar valores con frecuencia menor o igual a 5.
Teams = df['Team'].value_counts().sort_values(ascending=False)
Teams

Team
United States       13662
France               7790
Canada               7639
Great Britain        7494
Italy                7412
                    ...  
Greenoaks Dundee        1
Sunrise                 1
Dow Jones               1
Mexico-2                1
Digby                   1
Name: count, Length: 660, dtype: int64

Entrene un modelo de regresión lineal OLS. Reporte el error de entrenamiento y el error de prueba. Use RMSE como métrica de evaluación.

Reporte en una tabla los coeficientes asociados a cada característica del modelo.

Entrene un modelo Ridge. Sintonice  $\lambda$ por validación cruzada. Pruebe al menos 50 valores de  $\lambda$. 

Reporte el error promedio de validación cruzada, el valor de $\lambda$ sintonizado, el error de entrenamiento, y el error de prueba.

Reporte en una tabla los coeficientes asociados a cada característica del modelo.

Entrene un modelo Lasso. Sintonice  $\lambda$ por validación cruzada. Pruebe al menos 50 valores de  $\lambda$.

Reporte el error promedio de validación cruzada, el valor de $\lambda$ sintonizado, el error de entrenamiento, y el error de prueba.

Reporte en una tabla los coeficientes asociados a cada característica del modelo.

Use un modelo Lasso para hacer selección de características preservando únicamente 5 características en su modelo.

Con las características seleccionadas entrene nuevamente un modelo Ridge, sintonizando $\lambda$. 

Reporte el error promedio de validación cruzada, el valor de $\lambda$ sintonizado, el error de entrenamiento, y el error de prueba.

Reporte los coeficientes del modelo.